## Audit #1 — Record the class counts

In [1]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path("../Extracted")

class_counts = []

for class_dir in sorted(DATASET_DIR.iterdir()):
    if class_dir.is_dir():
        count = sum(1 for f in class_dir.rglob("*") if f.is_file())
        class_counts.append({
            "class": class_dir.name,
            "image_count": count
        })

class_counts_df = pd.DataFrame(class_counts)

class_counts_df.loc[len(class_counts_df)] = [
    "TOTAL",
    class_counts_df["image_count"].sum()
]

class_counts_df

,class,image_count
0,PESGL_DRECRO_exserohilum rostratum,11
1,PESGL_MOESBU_smut,97
2,PESGL_PYRISP_ Pyricularia sp,107
3,PESGL_SCLPGR_Sclerospora graminicola,31
4,PESGL_healthy leaves,41
5,TOTAL,287


## Audit #2 — File types

In [2]:
from collections import Counter

extensions = Counter()

for file_path in DATASET_DIR.rglob("*"):
    if file_path.is_file():
        suffix = file_path.suffix.lower()
        extensions[suffix] += 1

extension_df = pd.DataFrame(
    extensions.items(),
    columns=["extension", "count"]
).sort_values("count", ascending=False)

extension_df

,extension,count
0,.jpg,281
1,.pdf,4
2,,2


## Audit #3 — Inspect the actual file paths

In [3]:
sample_files = []

for file_path in DATASET_DIR.rglob("*"):
    if file_path.is_file():
        sample_files.append(str(file_path.relative_to(DATASET_DIR)))

sample_files[:30]

['PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_104.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_19.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_83.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_6.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_9.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_33.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_27.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_14.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_102.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_66.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_23.jpg',
 'PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_4.jpg',
 'PESGL_PYRISP_ P

## Audit #3 — Identify the 6 non-JPG files

In [4]:
non_jpg_files = [
    p for p in DATASET_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() != ".jpg"
]

non_jpg_df = pd.DataFrame({
    "path": [str(p.relative_to(DATASET_DIR)) for p in non_jpg_files],
    "extension": [p.suffix.lower() if p.suffix else "[NO EXTENSION]"
                  for p in non_jpg_files]
})

non_jpg_df

,path,extension
0,PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyr...,.pdf
1,PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCL...,.pdf
2,PESGL_DRECRO_exserohilum rostratum/PESGL_DRECR...,[NO EXTENSION]
3,PESGL_DRECRO_exserohilum rostratum/PESGL_DRECR...,.pdf
4,PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOES...,.pdf
5,PESGL_MOESBU_smut/PESGL_MOESBU_smut/.DS_Store,[NO EXTENSION]


In [5]:
for file_path in non_jpg_files:
    print("=" * 80)
    print(file_path)
    print(file_path.stat().st_size, "bytes")

../Extracted/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_suggested control measure.pdf
230520 bytes
../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_suggested control measure.pdf
215817 bytes
../Extracted/PESGL_DRECRO_exserohilum rostratum/PESGL_DRECRO_exserohilum rostratum/.DS_Store
6148 bytes
../Extracted/PESGL_DRECRO_exserohilum rostratum/PESGL_DRECRO_exserohilum rostratum/PESGL_DRECRO_suggested control measure.pdf
229827 bytes
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_suggested control measure.pdf
228570 bytes
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/.DS_Store
14340 bytes


In [6]:
print("Non-JPG files:", len(non_jpg_files))

Non-JPG files: 6


## Notebook — permanent audit record

In [8]:
from PIL import Image
from collections import Counter

image_files = list(DATASET_DIR.rglob("*.jpg"))

sizes = Counter()
modes = Counter()
errors = []

for p in image_files:
    try:
        with Image.open(p) as img:
            sizes[img.size] += 1
            modes[img.mode] += 1
    except Exception as e:
        errors.append((str(p), str(e)))

print("Total JPG images:", len(image_files))

print("\nImage modes:")
for mode, count in modes.items():
    print(f"{mode}: {count}")

print("\nUnique dimensions:", len(sizes))

print("\nTop 20 dimensions:")
for size, count in sizes.most_common(20):
    print(f"{size}: {count}")

print("\nImage read errors:", len(errors))

Total JPG images: 281

Image modes:
RGB: 281

Unique dimensions: 18

Top 20 dimensions:
(4000, 3000): 174
(780, 1040): 48
(768, 1024): 24
(3000, 4000): 11
(472, 1024): 6
(1040, 780): 3
(2592, 4608): 2
(591, 1280): 2
(960, 1280): 2
(457, 1024): 1
(620, 1024): 1
(623, 1024): 1
(830, 1280): 1
(664, 1024): 1
(947, 1088): 1
(538, 1024): 1
(926, 1017): 1
(769, 802): 1

Image read errors: 0


## check for duplicate images

In [9]:
import hashlib
from collections import defaultdict

image_files = list(DATASET_DIR.rglob("*.jpg"))

hashes = defaultdict(list)

for p in image_files:
    h = hashlib.md5(p.read_bytes()).hexdigest()
    hashes[h].append(str(p))

duplicate_groups = {
    h: files for h, files in hashes.items()
    if len(files) > 1
}

print("Total JPG images:", len(image_files))
print("Unique image hashes:", len(hashes))
print("Exact duplicate groups:", len(duplicate_groups))
print(
    "Duplicate image files:",
    sum(len(files) - 1 for files in duplicate_groups.values())
)

for h, files in duplicate_groups.items():
    print("\nHASH:", h)
    for f in files:
        print(" ", f)

Total JPG images: 281
Unique image hashes: 264
Exact duplicate groups: 17
Duplicate image files: 17

HASH: d7cfcfca46ce7075304e60b0ce1e25c0
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_31.jpg
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_10.jpg

HASH: 8c6ebaec64c5a8668ab143ebd8ebb4b8
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_41.jpg
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_21.jpg

HASH: 9a249e381363c738bfb1a3627857d4fc
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_39.jpg
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_18.jpg

HASH: f0ad3f2910d2e15324a376791d73ce30
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_14.jpg
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_35.jpg

HASH: e333b085b62639076d2a47a4ee1064f8
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_12.jpg
  ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_33.jpg


## remove exact duplicates

In [10]:
# Exact duplicate cleaning record

duplicate_groups_found = 17
duplicate_files_removed = 17
images_before_cleaning = 281
images_after_cleaning = 264

print("Exact duplicate groups found:", duplicate_groups_found)
print("Duplicate files removed:", duplicate_files_removed)
print("Images before cleaning:", images_before_cleaning)
print("Images after cleaning:", images_after_cleaning)

Exact duplicate groups found: 17
Duplicate files removed: 17
Images before cleaning: 281
Images after cleaning: 264


## class distribution

In [12]:
from collections import Counter

class_counts = {}

for class_dir in DATASET_DIR.iterdir():
    if class_dir.is_dir():
        count = len(list(class_dir.rglob("*.jpg")))
        class_counts[class_dir.name] = count

print("Class distribution:")
print("-" * 60)

total = 0
for name, count in sorted(class_counts.items()):
    print(f"{name}: {count}")
    total += count

print("-" * 60)
print("Total JPG images:", total)

Class distribution:
------------------------------------------------------------
PESGL_DRECRO_exserohilum rostratum: 9
PESGL_MOESBU_smut: 95
PESGL_PYRISP_ Pyricularia sp: 106
PESGL_SCLPGR_Sclerospora graminicola: 30
PESGL_healthy leaves: 24
------------------------------------------------------------
Total JPG images: 264


## terminal command: for duplicate remove
###
 python - <<'PY'
from pathlib import Path
from collections import defaultdict
import hashlib

root = Path("Extracted")
hashes = defaultdict(list)

for p in root.rglob("*.jpg"):
    h = hashlib.md5(p.read_bytes()).hexdigest()
    hashes[h].append(p)

deleted = []

for files in hashes.values():
    if len(files) > 1:
        for p in files[1:]:
            p.unlink()
            deleted.append(p)

print("Deleted:", len(deleted))
print("Remaining JPG:", len(list(root.rglob("*.jpg"))))
PY

In [15]:
from PIL import Image
import imagehash

image_files = list(DATASET_DIR.rglob("*.jpg"))

image_hashes = {}

for p in image_files:
    with Image.open(p) as img:
        image_hashes[p] = imagehash.phash(img)

threshold = 4
near_duplicate_pairs = []

files = list(image_hashes.keys())

for i in range(len(files)):
    for j in range(i + 1, len(files)):
        distance = image_hashes[files[i]] - image_hashes[files[j]]

        if distance <= threshold:
            near_duplicate_pairs.append(
                (distance, files[i], files[j])
            )

near_duplicate_pairs.sort(key=lambda x: x[0])

print("Images checked:", len(image_files))
print(
    "Near-duplicate pairs (pHash distance <= 4):",
    len(near_duplicate_pairs)
)

for distance, a, b in near_duplicate_pairs:
    print(f"\nDistance: {distance}")
    print("A:", a)
    print("B:", b)

Images checked: 264
Near-duplicate pairs (pHash distance <= 4): 39

Distance: 0
A: ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_15.jpg
B: ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_16.jpg

Distance: 0
A: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_15.jpg
B: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_10.jpg

Distance: 0
A: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_17.jpg
B: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_14.jpg

Distance: 0
A: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_13.jpg
B: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_19.jpg

Distance: 0
A: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_16.jpg
B: ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_11.jpg

Distance: 0
A: ../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/P

In [16]:
from PIL import Image, ImageDraw
import math

# Show all near-duplicate pairs found above
pairs = near_duplicate_pairs

thumb_w = 300
thumb_h = 250
label_h = 55

cols = 2
rows = len(pairs)

sheet = Image.new(
    "RGB",
    (cols * thumb_w, rows * thumb_h),
    "white"
)

draw = ImageDraw.Draw(sheet)

for i, (distance, a, b) in enumerate(pairs):
    y = i * thumb_h

    with Image.open(a) as img_a:
        img_a = img_a.convert("RGB")
        img_a.thumbnail((thumb_w - 20, thumb_h - label_h - 10))

    with Image.open(b) as img_b:
        img_b = img_b.convert("RGB")
        img_b.thumbnail((thumb_w - 20, thumb_h - label_h - 10))

    x1 = 0
    x2 = thumb_w

    sheet.paste(
        img_a,
        (
            x1 + (thumb_w - img_a.width) // 2,
            y + 5
        )
    )

    sheet.paste(
        img_b,
        (
            x2 + (thumb_w - img_b.width) // 2,
            y + 5
        )
    )

    draw.text(
        (10, y + thumb_h - label_h),
        f"Distance {distance}\nA: {a.name}",
        fill="black"
    )

    draw.text(
        (thumb_w + 10, y + thumb_h - label_h),
        f"Distance {distance}\nB: {b.name}",
        fill="black"
    )

audit_dir = DATASET_DIR / "Audit"
audit_dir.mkdir(exist_ok=True)

output_path = audit_dir / "near_duplicate_contact_sheet.jpg"
sheet.save(output_path, quality=95)

print("Saved:", output_path)
print("Pairs shown:", len(pairs))

Saved: ../Extracted/Audit/near_duplicate_contact_sheet.jpg
Pairs shown: 39


In [17]:
from PIL import Image, ImageDraw

def make_contact_sheet(pairs, start, end, filename):
    selected = pairs[start:end]

    thumb_w = 300
    row_h = 260
    cols = 2

    sheet = Image.new(
        "RGB",
        (cols * thumb_w, len(selected) * row_h),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for row, (distance, a, b) in enumerate(selected):
        y = row * row_h

        for col, path in enumerate([a, b]):
            with Image.open(path) as img:
                img = img.convert("RGB")
                img.thumbnail((280, 190))

                x = col * thumb_w
                sheet.paste(
                    img,
                    (
                        x + (thumb_w - img.width) // 2,
                        y + 5
                    )
                )

        draw.text(
            (10, y + 205),
            f"Distance: {distance} | {a.name}",
            fill="black"
        )

        draw.text(
            (310, y + 205),
            f"Distance: {distance} | {b.name}",
            fill="black"
        )

    path = DATASET_DIR / "Audit" / filename
    sheet.save(path, quality=95)

    print("Saved:", path)


make_contact_sheet(
    near_duplicate_pairs,
    0,
    10,
    "near_duplicates_01.jpg"
)

make_contact_sheet(
    near_duplicate_pairs,
    10,
    20,
    "near_duplicates_02.jpg"
)

make_contact_sheet(
    near_duplicate_pairs,
    20,
    30,
    "near_duplicates_03.jpg"
)

make_contact_sheet(
    near_duplicate_pairs,
    30,
    39,
    "near_duplicates_04.jpg"
)

Saved: ../Extracted/Audit/near_duplicates_01.jpg
Saved: ../Extracted/Audit/near_duplicates_02.jpg
Saved: ../Extracted/Audit/near_duplicates_03.jpg
Saved: ../Extracted/Audit/near_duplicates_04.jpg


## Build the confirmed duplicate groups

In [18]:
from collections import defaultdict

# Your confirmed near-duplicate pairs
# Format: (pHash distance, image_A, image_B)
confirmed_pairs = near_duplicate_pairs  # all 39 pairs you already generated

# Build graph
duplicate_graph = defaultdict(set)

for distance, a, b in confirmed_pairs:
    duplicate_graph[a].add(b)
    duplicate_graph[b].add(a)

# Find connected duplicate groups
visited = set()
duplicate_groups = []

for image in duplicate_graph:
    if image in visited:
        continue

    stack = [image]
    group = set()

    while stack:
        current = stack.pop()

        if current in visited:
            continue

        visited.add(current)
        group.add(current)

        for neighbor in duplicate_graph[current]:
            if neighbor not in visited:
                stack.append(neighbor)

    duplicate_groups.append(sorted(group))

print("Confirmed duplicate groups:", len(duplicate_groups))
print("Images involved:", sum(len(g) for g in duplicate_groups))
print("Files that would be removed:", sum(len(g) - 1 for g in duplicate_groups))

print("\nDuplicate groups:")
for i, group in enumerate(duplicate_groups, 1):
    print(f"\nGroup {i} ({len(group)} images):")
    for img in group:
        print(" ", img)

Confirmed duplicate groups: 30
Images involved: 66
Files that would be removed: 36

Duplicate groups:

Group 1 (4 images):
  ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_15.jpg
  ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_16.jpg
  ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_17.jpg
  ../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_18.jpg

Group 2 (2 images):
  ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_10.jpg
  ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_15.jpg

Group 3 (2 images):
  ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_14.jpg
  ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_17.jpg

Group 4 (2 images):
  ../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_13.jpg
  ../Extracted/PESGL_MO

## duplicate_groups

In [19]:
from pathlib import Path

# Keep the first image from each confirmed duplicate group.
# Delete every other image.

files_to_delete = []

for group in duplicate_groups:
    # Keep first
    keep = group[0]

    # Delete remaining
    for duplicate in group[1:]:
        files_to_delete.append(Path(duplicate))

print("Files to delete:", len(files_to_delete))
print()

for path in files_to_delete:
    print(path)

Files to delete: 36

../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_16.jpg
../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_17.jpg
../Extracted/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_Sclerospora graminicola/PESGL_SCLPGR_18.jpg
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_15.jpg
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_17.jpg
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_19.jpg
../Extracted/PESGL_MOESBU_smut/PESGL_MOESBU_smut/PESGL_MOESBU_16.jpg
../Extracted/PESGL_healthy leaves/PESGL_healthy leaves/PESGL_23.jpg
../Extracted/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_9.jpg
../Extracted/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_54.jpg
../Extracted/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_ Pyricularia sp/PESGL_PYRISP_59.jpg
../Extracted/PESGL_PYRISP_ Pyricularia sp/

In [20]:
deleted = 0
missing = 0

for path in files_to_delete:
    if path.exists():
        path.unlink()
        deleted += 1
    else:
        missing += 1

print("Deleted:", deleted)
print("Already missing:", missing)
print("Expected remaining JPG images:", 264 - deleted)

Deleted: 36
Already missing: 0
Expected remaining JPG images: 228


In [21]:
deleted = 0
missing = 0

for path in files_to_delete:
    if path.exists():
        path.unlink()
        deleted += 1
    else:
        missing += 1

print("Deleted:", deleted)
print("Already missing:", missing)

Deleted: 0
Already missing: 36


In [22]:
#re-verification
from pathlib import Path
from collections import Counter

root = Path("../Extracted")

jpg_files = list(root.rglob("*.jpg"))

print("Remaining JPG images:", len(jpg_files))

# Class distribution based on parent folder
class_counts = Counter()

for path in jpg_files:
    class_name = path.parts[-3]
    class_counts[class_name] += 1

print("\nClass distribution:")
for cls, count in sorted(class_counts.items()):
    print(f"{cls}: {count}")

Remaining JPG images: 233

Class distribution:
Extracted: 5
PESGL_DRECRO_exserohilum rostratum: 9
PESGL_MOESBU_smut: 75
PESGL_PYRISP_ Pyricularia sp: 98
PESGL_SCLPGR_Sclerospora graminicola: 23
PESGL_healthy leaves: 23


In [23]:
#extra checking
from pathlib import Path

root = Path("../Extracted")

jpg_files = list(root.rglob("*.jpg"))

extra = []

valid_classes = {
    "PESGL_DRECRO_exserohilum rostratum",
    "PESGL_MOESBU_smut",
    "PESGL_PYRISP_ Pyricularia sp",
    "PESGL_SCLPGR_Sclerospora graminicola",
    "PESGL_healthy leaves"
}

for path in jpg_files:
    # Find which class directory contains the image
    if not any(cls in path.parts for cls in valid_classes):
        extra.append(path)

print("Extra JPG files:", len(extra))

for path in extra:
    print(path)

Extra JPG files: 5
../Extracted/Audit/near_duplicates_04.jpg
../Extracted/Audit/near_duplicates_03.jpg
../Extracted/Audit/near_duplicates_01.jpg
../Extracted/Audit/near_duplicates_02.jpg
../Extracted/Audit/near_duplicate_contact_sheet.jpg


## Phase 2 — Dataset Quality & Provenance Audit

In [24]:
##Step 2.1 — Verify the final class structure
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image
import pandas as pd

root = Path("../Extracted")

class_dirs = [
    root / "PESGL_DRECRO_exserohilum rostratum",
    root / "PESGL_MOESBU_smut",
    root / "PESGL_PYRISP_ Pyricularia sp",
    root / "PESGL_SCLPGR_Sclerospora graminicola",
    root / "PESGL_healthy leaves",
]

records = []

for class_dir in class_dirs:
    class_name = class_dir.name

    # JPG files only
    images = list(class_dir.rglob("*.jpg"))

    for img_path in images:
        try:
            with Image.open(img_path) as img:
                records.append({
                    "class": class_name,
                    "file": str(img_path),
                    "width": img.width,
                    "height": img.height,
                    "mode": img.mode,
                    "format": img.format,
                    "readable": True
                })
        except Exception as e:
            records.append({
                "class": class_name,
                "file": str(img_path),
                "width": None,
                "height": None,
                "mode": None,
                "format": None,
                "readable": False
            })

df_audit = pd.DataFrame(records)

print("Total dataset images:", len(df_audit))
print("Readable:", df_audit["readable"].sum())
print("Unreadable:", (~df_audit["readable"]).sum())

print("\nClass distribution:")
print(df_audit["class"].value_counts())

print("\nImage modes:")
print(df_audit["mode"].value_counts())

print("\nImage dimensions:")
print(
    df_audit
    .groupby("class")[["width", "height"]]
    .agg(["min", "max", "count"])
)

Total dataset images: 228
Readable: 228
Unreadable: 0

Class distribution:
class
PESGL_PYRISP_ Pyricularia sp            98
PESGL_MOESBU_smut                       75
PESGL_SCLPGR_Sclerospora graminicola    23
PESGL_healthy leaves                    23
PESGL_DRECRO_exserohilum rostratum       9
Name: count, dtype: int64

Image modes:
mode
RGB    228
Name: count, dtype: int64

Image dimensions:
                                     width             height            
                                       min   max count    min   max count
class                                                                    
PESGL_DRECRO_exserohilum rostratum     780  4000     9    780  4608     9
PESGL_MOESBU_smut                      472  4000    75    802  3000    75
PESGL_PYRISP_ Pyricularia sp          3000  4000    98   3000  4000    98
PESGL_SCLPGR_Sclerospora graminicola   457   780    23   1024  1040    23
PESGL_healthy leaves                   780   780    23   1040  1040    23


In [25]:
#Phase 2.2 — Check whether images come from repeated physical samples
#Step 2.2A — Create class-wise contact sheets
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import math

root = Path("../Extracted")
audit_dir = root / "Audit" / "class_contact_sheets"
audit_dir.mkdir(parents=True, exist_ok=True)

class_dirs = [
    root / "PESGL_DRECRO_exserohilum rostratum",
    root / "PESGL_MOESBU_smut",
    root / "PESGL_PYRISP_ Pyricularia sp",
    root / "PESGL_SCLPGR_Sclerospora graminicola",
    root / "PESGL_healthy leaves",
]

THUMB_W = 220
THUMB_H = 180
LABEL_H = 35
COLS = 5

for class_dir in class_dirs:

    images = sorted(class_dir.rglob("*.jpg"))

    rows = math.ceil(len(images) / COLS)

    sheet = Image.new(
        "RGB",
        (COLS * THUMB_W, rows * (THUMB_H + LABEL_H)),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for i, img_path in enumerate(images):

        try:
            img = Image.open(img_path).convert("RGB")
            img.thumbnail((THUMB_W - 10, THUMB_H - 10))

            x = (i % COLS) * THUMB_W
            y = (i // COLS) * (THUMB_H + LABEL_H)

            px = x + (THUMB_W - img.width) // 2
            py = y + (THUMB_H - img.height) // 2

            sheet.paste(img, (px, py))

            label = img_path.stem
            draw.text(
                (x + 5, y + THUMB_H),
                label,
                fill="black"
            )

        except Exception as e:
            print("Error:", img_path, e)

    output = audit_dir / f"{class_dir.name}_contact_sheet.jpg"
    sheet.save(output, quality=90)

    print(f"Created: {output}")

Created: ../Extracted/Audit/class_contact_sheets/PESGL_DRECRO_exserohilum rostratum_contact_sheet.jpg
Created: ../Extracted/Audit/class_contact_sheets/PESGL_MOESBU_smut_contact_sheet.jpg
Created: ../Extracted/Audit/class_contact_sheets/PESGL_PYRISP_ Pyricularia sp_contact_sheet.jpg
Created: ../Extracted/Audit/class_contact_sheets/PESGL_SCLPGR_Sclerospora graminicola_contact_sheet.jpg
Created: ../Extracted/Audit/class_contact_sheets/PESGL_healthy leaves_contact_sheet.jpg


In [ ]:
#Step 2.2B — Inspect for repeated physical specimens
path = r"../Extracted/Audit/class_contact_sheets/"

In [26]:
specimen_audit = {
    "method": "Manual visual inspection of class-wise contact sheets",
    "classes_inspected": 5,
    "obvious_repeated_specimens_found": False,
    "notes": (
        "Images were compared for identical leaf shape, lesion pattern, "
        "damage, orientation, background, and framing. "
        "No obvious repeated physical specimens were identified."
    )
}

specimen_audit

{'method': 'Manual visual inspection of class-wise contact sheets',
 'classes_inspected': 5,
 'obvious_repeated_specimens_found': False,
 'notes': 'Images were compared for identical leaf shape, lesion pattern, damage, orientation, background, and framing. No obvious repeated physical specimens were identified.'}

In [ ]:
# #Step 2.3 — Dataset provenance/documentation audit.
# That is important because we need to record things like:

# dataset source
# Zenodo record
# collection location
# collection year/date if available
# disease labels
# healthy label
# number of original images
# whether images were collected in Burkina Faso
# license
# citation
# whether the classes are biologically comparable
# limitations of the dataset

In [ ]:
# #Phase 2.3: Provenance & Documentation Audit.
# We'll verify the official metadata from Zenodo and record it in your notebook, including:

# dataset title and DOI
# creator/organization
# collection location
# crop/species
# disease classes
# license/rights
# original dataset structure
# our cleaning operations and final counts

# The official record gives DOI 10.5281/zenodo.17086194 and identifies ACRA as the copyright holder; it also lists the dataset as collected in Burkina Faso under the EWA-BELT/PLANTHEAD work.
# https://zenodo.org/records/17086194?utm_source=chatgpt.com